In [ ]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-user-graduate" style="color:#18ab4b"></i> BYOC training — CatBoost credit model

Build a <i class="fas fa-file-code" style="color:#6c63ff"></i> **bring-your-own-container** (CatBoost), train it with SageMaker local mode (`instance_type='local'` — a real Docker training job), and store the <i class="fas fa-brain" style="color:#18ab4b"></i> model in oblako's <i class="fas fa-database" style="color:#18ab4b"></i> **S3Proxy**. Mirrors `sagemaker_endpoints/ml_models`.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## <i class="fas fa-file-code" style="color:#6c63ff"></i> Build the container + <i class="fas fa-user-graduate" style="color:#18ab4b"></i> train

In [ ]:
import os
import tempfile

import creditdata
from sagemaker.estimator import Estimator

from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

IMAGE = "oblako-credit-risk:latest"
HP = {
    "iterations": "150",
    "depth": "6",
    "learning-rate": "0.1",
    "factor": "20",
    "offset": "600",
}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # CatBoost BYOC (./container)

data_dir = tempfile.mkdtemp(prefix="credit-train-")
creditdata.write_training_csv(os.path.join(data_dir, "train.csv"), n=600)
out_dir = tempfile.mkdtemp(prefix="credit-out-")

estimator = Estimator(
    image_uri=IMAGE,
    role="arn:aws:iam::000000000000:role/dummy",
    instance_count=1,
    instance_type="local",
    sagemaker_session=sm.get_session(),
    output_path=f"file://{out_dir}",
    hyperparameters=HP,
)
estimator.fit({"train": f"file://{data_dir}"})

## <i class="fas fa-database" style="color:#18ab4b"></i> Store the trained model in oblako's S3

In [ ]:
import glob

from oblako.services import S3ProxyService

model_tar = glob.glob(os.path.join(out_dir, "**", "model.tar.gz"), recursive=True)[0]
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="credit-risk-models")
except Exception:
    pass
s3.upload_file(model_tar, "credit-risk-models", "credit-risk/model.tar.gz")
print("stored: s3://credit-risk-models/credit-risk/model.tar.gz")
print(
    "bucket:",
    [
        o["Key"]
        for o in s3.list_objects_v2(Bucket="credit-risk-models").get("Contents", [])
    ],
)